In [3]:
from rima_pfn.data import RIMADataset

d = RIMADataset.build().d

In [4]:
d.head()

,album_id,album_name,album_release_date,album_type,total_tracks,label,album_popularity,release_type,artist_id,artist_name,...,hpcp_8_mean,hpcp_9_mean,hpcp_10_mean,hpcp_11_mean,is_male,is_female,is_group,is_album,is_single,is_compilation
0,ALB040c17efb,#Rossoneri,2015-09-11,single,1,Carosello Records,21,None,ART03d65f2cd,emis killa,...,0.234467,0.274434,0.544141,0.338240,True,False,False,False,True,False
1,ALB040c17efb,#Rossoneri,2015-09-11,single,1,Carosello Records,21,None,ART03d65f2cd,emis killa,...,0.460163,0.237569,0.198111,0.235746,True,False,False,False,True,False
2,ALB040c17efb,#Rossoneri,2015-09-11,single,1,Carosello Records,21,None,ART03d65f2cd,emis killa,...,0.434683,0.270943,0.203755,0.184407,True,False,False,False,True,False
3,ALB040c17efb,#Rossoneri,2015-09-11,single,1,Carosello Records,21,None,ART03d65f2cd,emis killa,...,0.301894,0.169535,0.195710,0.244809,True,False,False,False,True,False
4,ALB040c17efb,#Rossoneri,2015-09-11,single,1,Carosello Records,21,None,ART03d65f2cd,emis killa,...,0.181203,0.544622,0.291527,0.220556,True,False,False,False,True,False


In [5]:
d.select_dtypes(include="number").head()

,total_tracks,album_popularity,birth_lat,birth_lon,release_year,release_month,release_day,stats_pageviews,bpm,beats_count,...,hpcp_2_mean,hpcp_3_mean,hpcp_4_mean,hpcp_5_mean,hpcp_6_mean,hpcp_7_mean,hpcp_8_mean,hpcp_9_mean,hpcp_10_mean,hpcp_11_mean
0,1,21,45.616667,9.366667,2011.0,4.0,2.0,NaN,86.842186,182.0,...,0.169642,0.380168,0.169968,0.210395,0.306807,0.296305,0.234467,0.274434,0.544141,0.338240
1,1,21,45.616667,9.366667,2021.0,7.0,23.0,26726.0,139.929108,302.0,...,0.215512,0.287880,0.419228,0.153843,0.207034,0.196229,0.460163,0.237569,0.198111,0.235746
2,1,21,45.616667,9.366667,2018.0,1.0,10.0,39517.0,92.732407,284.0,...,0.111401,0.315526,0.297582,0.135717,0.311567,0.183751,0.434683,0.270943,0.203755,0.184407
3,1,21,45.616667,9.366667,2018.0,10.0,12.0,36091.0,120.022804,384.0,...,0.265708,0.304442,0.159810,0.260745,0.181525,0.378647,0.301894,0.169535,0.195710,0.244809
4,1,21,45.616667,9.366667,2016.0,10.0,14.0,22096.0,116.740265,439.0,...,0.301903,0.143441,0.430993,0.273239,0.129572,0.244877,0.181203,0.544622,0.291527,0.220556


In [13]:
import pandas
from rima_pfn.data import CONFIGS
from datasets import load_dataset

rima_dict = {
    config: load_dataset("mstz/rima", config)["train"].to_pandas()
    for config in CONFIGS
}
merge = pandas.merge(
    rima_dict["albums"],
    rima_dict["artists"],
    left_on="artist_ids",
    right_on="id_artist",
    suffixes=("_album", "_artist"),
)
merge = pandas.merge(
    merge,
    rima_dict["audio"],
    left_on="name",
    right_on="name_artist",
    suffixes=("_merge1", "_audio"),
)
merge.drop([
    "id_album_spotify",
    "artist_ids",
    "artist_names",
    "upc",
    "mbid_release",
    "release_country",
    "barcode",
    "qid",
    "image_url",
    "mbid",
    "mb_type",
    "mb_area",
    "mb_begin_area",
    "mb_aliases",
    "id_track_genius",
    "id_artist_genius",
    "name_artist_genius",
    "featured_artists",
    "primary_artist",
    "language",
    "lyrics",
    "url",
    "lang_langdetect",
    "lang_langdetect_prob",
    "lang_langid",
    "lang_langid_score",
    "lang_lingua",
    "lang_lingua_conf",
    "id_artist_audio",
    "__index_level_0__",
    "album_image",
    "active_end",
    ],
    axis="columns",
    inplace=True,
)
merge.rename({
    "id_artist_merge1": "artist_id",
    "id_album": "album_id",
    "name": "artist_name",
    "id_track": "track_id",
    },
    axis="columns",
    inplace=True,
)
merge.columns


Index(['album_id', 'album_name', 'album_release_date', 'album_type',
       'total_tracks', 'label', 'album_popularity', 'release_type',
       'artist_id', 'artist_name',
       ...
       'hpcp_2_mean', 'hpcp_3_mean', 'hpcp_4_mean', 'hpcp_5_mean',
       'hpcp_6_mean', 'hpcp_7_mean', 'hpcp_8_mean', 'hpcp_9_mean',
       'hpcp_10_mean', 'hpcp_11_mean'],
      dtype='object', length=145)

In [16]:
len(d.columns), len(merge.columns)

(151, 145)

In [22]:
d = RIMADataset.build().d

###########################################################
# region/school classification ############################
###########################################################
df = d.select_dtypes(include="number").drop(columns=[
    "total_tracks",
    "album_popularity",
    "birth_lat",
    "birth_lon",
])
df = pandas.concat([df, d[["birth_region"]]], axis="columns")
df = df[~df["birth_region"].isna()]

In [33]:
from sklearn.metrics import classification_report
y_true = [0, 1, 2, 2, 2]
y_pred = [0, 0, 2, 2, 1]
target_names = ['class 0', 'class 1', 'class 2']
classification_report(y_true, y_pred, target_names=target_names, output_dict=True)["macro avg"]

{'precision': 0.5,
 'recall': 0.5555555555555555,
 'f1-score': 0.48888888888888893,
 'support': 5.0}